# FPGA 卷积＋并行运动监测 · DeepSeek

独立工程：`fpga_pose_parallel_ai_20261028`。原工程仍保留。此 Notebook 只检查状态，不重复加载 FPGA。


## 1. 文件检查


In [ ]:
from pathlib import Path
import sys, json
PROJECT = Path('/home/xilinx/jupyter_notebooks/fpga_pose_parallel_ai_20261028')
assert PROJECT.is_dir(), '请先上传并行版工程'
for name in ('overlays/pose_distance.bit', 'overlays/pose_distance.hwh', 'parallel_version.json'):
    assert (PROJECT/name).is_file(), '缺少文件：' + name
sys.path.insert(0, str(PROJECT))
from fpga_image import validate_v2_hwh
validate_v2_hwh(PROJECT/'overlays/pose_distance.hwh')
print('文件检查通过｜卷积版｜8 项运动')


## 2. 运行状态


In [ ]:
from urllib.request import build_opener, ProxyHandler
opener = build_opener(ProxyHandler({}))
with opener.open('http://127.0.0.1:8081/api/status', timeout=5) as response:
    state = json.load(response)
assert state.get('project_version') == 'parallel-ai-conv-20261028', '当前不是卷积版；请查看“卷积版使用说明.md”'
assert state['ready'] and state['running'], '服务未就绪，请查看新服务日志'
assert state['hardware']['parallel_lanes'] == 32
assert state['convolution']['ready'] and state['convolution']['self_test_passed']
print('FPGA：36 路卷积＋32 路距离｜摄像头：运行')
print('http://192.168.2.99:8081/?v=parallel-ai-conv-20261028')


## 3. AI 连接


In [ ]:
with opener.open('http://127.0.0.1:8081/api/status', timeout=5) as response:
    state = json.load(response)
ai = state['ai']
print('DeepSeek 助手：' + ('已连接' if ai.get('relay_connected') else '等待电脑助手'))
print('网页 AI 点评：' + ('已启用' if ai.get('enabled') else '未启用'))

print('电脑卷积助手：' + ('已连接' if state.get('pose_relay', {}).get('relay_connected') else '等待连接'))


新旧工程切换、重新构建和对比测试命令见同目录 **卷积版使用说明.md**。训练前选择动作并部署模型，网页勾选语音或 AI 后使用。
